# Hold maps from labeled position stills

Turns the labeled position stills (stage 2 Roboflow project, COCO Segmentation export) into:
- `hold-maps/holds.csv`: one row per hold (position, hold ID, class, color, center, bounding box, area)
- `hold-maps/hold_polygons.json`: each hold's outline, for contact extraction and cropping later
- `hold-maps/posX_numbered.png`: each still with every hold outlined and numbered
- `hold-maps/routes.csv`: each route's holds, pre-filled by color, to finish by hand
- `hold-maps/posX_frame.png`: the clean empty-wall video frame used for each position (reference image in video coordinates)
- `hold-maps/checks/posX_overlay.jpg`: converted outlines drawn on the video's first frame, to check alignment

The stills are phone screenshots of the videos, so their pixels don't line up with the video's. Cell 4 finds where each screenshot sits inside its video frame and every outline is converted to **video coordinates**, which is what the pose results use.

## Cell 1: Setup

In [ ]:
!pip install -q roboflow
import os, re, json, glob
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from google.colab import drive, userdata

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ECS-174-Project'
VIDEO_DIR = f'{BASE}/original-footage'
MAP_DIR = f'{BASE}/hold-maps'
META_CSV = f'{BASE}/metadata.csv'
VIDEO_COL, POS_COL, COLOR_COL = 'Filename', 'Position_ID', 'Route Grade/Color'
START_COL = 'Climb_Start'                   # seconds; frames before this are the empty wall

# Positions left out entirely (camera too shaky to align, or no matching videos)
EXCLUDE_POSITIONS = {'pos2', 'pos4', 'pos6', 'pos7', 'pos11', 'pos18', 'pos22', 'pos25'}
VIDEO_EXTS = ('.mp4', '.mov', '.m4v', '.avi')

os.makedirs(f'{MAP_DIR}/checks', exist_ok=True)

def normalize_pos(s):
    """'Pos02', 'pos2', 'pos02_still' -> 'pos2'."""
    m = re.search(r'pos0*(\d+)', str(s), re.IGNORECASE)
    return f'pos{int(m.group(1))}' if m else None

## Cell 2: Download the COCO export

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("bouldering-hold-labeling").project("bouldering-holds-stage-2")
dataset = project.version(3).download("coco-segmentation")
print(dataset.location)

## Cell 3: Load the labels and match stills to positions
Stills without a position in their name are skipped.
Should report one still per position.

In [ ]:
images, anns_by_img, cats = {}, defaultdict(list), {}

for coco_file in glob.glob(f'{dataset.location}/*/_annotations.coco.json'):
    folder = os.path.dirname(coco_file)
    coco = json.load(open(coco_file))
    for c in coco['categories']:
        cats[c['id']] = c['name']
    for im in coco['images']:
        key = (coco_file, im['id'])            # image ids restart in each split's file
        images[key] = {**im, 'path': os.path.join(folder, im['file_name'])}
    for a in coco['annotations']:
        anns_by_img[(coco_file, a['image_id'])].append(a)

def position_of(file_name):
    """Roboflow renames 'pos3.jpg' to 'pos3_jpg.rf.<hash>.jpg'; recover 'pos3'."""
    return normalize_pos(file_name.split('.rf.')[0])

seen = {}
for key, im in images.items():
    im['pos'] = position_of(im['file_name'])
    if im['pos'] is None:
        print('Skipping (no position in filename):', im['file_name'])
    elif im['pos'] in seen:
        print(f"WARNING: two stills for {im['pos']}: {seen[im['pos']]} and {im['file_name']}")
    seen[im['pos']] = im['file_name']

images = {k: im for k, im in images.items() if im['pos']}      # drop stills that aren't positions
excluded = sorted(im['pos'] for im in images.values() if im['pos'] in EXCLUDE_POSITIONS)
images = {k: im for k, im in images.items() if im['pos'] not in EXCLUDE_POSITIONS}
print('Excluded positions:', excluded)

counts = Counter(cats[a['category_id']] for key in images for a in anns_by_img[key])
print(f'{len(images)} position stills, {sum(counts.values())} labeled objects')
print('Objects per class:', dict(sorted(counts.items())))

## Cell 4: Metadata, route IDs, and aligning stills to the videos
- Builds route IDs from filenames: `izzy-green-1` and `izzy-green-1.2` (attempt 2) at pos25 both become `green1`
- Positions in `EXCLUDE_POSITIONS` (Cell 1) are dropped, along with their videos and routes.
- For each position, reads frames from the empty-wall part of the video (up to 0.5 s before `Climb_Start`), picks the one that best matches the screenshot, and finds the **scale and offset** that place the screenshot inside it (template matching). `frame_used` says which frame.
- `match_score` near 1 means a confident match.

In [ ]:
meta = pd.read_csv(META_CSV)

def strip_video_ext(name):
    """Remove a real video extension only"""
    name = str(name)
    for ext in VIDEO_EXTS:
        if name.lower().endswith(ext):
            return name[:-len(ext)]
    return name

def route_id_of(row):
    """'izzy-green-1' or 'izzy-green-1.2' -> 'green1' (attempt number dropped)."""
    parts = strip_video_ext(row[VIDEO_COL]).lower().split('-')
    color_word, number = parts[-2], parts[-1].split('.')[0]
    return f"{color_word}{number}"

dropped = meta[meta[POS_COL].isin(EXCLUDE_POSITIONS)]
meta = meta[~meta[POS_COL].isin(EXCLUDE_POSITIONS)].copy()
meta['route_id'] = meta.apply(route_id_of, axis=1)
print(f'Excluded positions remove {len(dropped)} videos; {len(meta)} videos remain at '
      f'{meta[POS_COL].nunique()} positions')
meta['route_color'] = meta[COLOR_COL].astype(str).str.lower()
print(meta[[VIDEO_COL, 'route_id', 'route_color']].head(10))

# Exact filename matching: 'izzy-green-1' must not match 'izzy-green-1.2.mp4'
video_files = {os.path.splitext(f)[0]: os.path.join(VIDEO_DIR, f)
               for f in os.listdir(VIDEO_DIR) if f.lower().endswith(VIDEO_EXTS)}

def find_video(name):
    return video_files.get(strip_video_ext(name))

climb_start = {strip_video_ext(r[VIDEO_COL]): r[START_COL] for _, r in meta.iterrows()}

def rle_counts(s):
    """Decode COCO's compressed RLE string into run lengths (same algorithm as pycocotools)."""
    counts, p = [], 0
    while p < len(s):
        x, k, more = 0, 0, True
        while more:
            c = ord(s[p]) - 48
            x |= (c & 0x1f) << (5 * k)
            more = bool(c & 0x20)
            p += 1
            k += 1
            if not more and (c & 0x10):
                x |= -1 << (5 * k)
        if len(counts) > 2:
            x += counts[-2]
        counts.append(x)
    return counts

def rle_to_mask(seg):
    """RLE mask -> binary image. Runs alternate 0s and 1s, going down columns (column-major)."""
    h, w = seg['size']
    counts = seg['counts'] if isinstance(seg['counts'], list) else rle_counts(seg['counts'])
    flat = np.zeros(h * w, dtype=np.uint8)
    pos, val = 0, 0
    for c in counts:
        if val:
            flat[pos:pos + c] = 1
        pos += c
        val ^= 1
    return flat.reshape((h, w), order='F')

def polygon_parts(seg):
    """Outline(s) of one label as point arrays. Handles both COCO formats:
    polygons (flat lists [x1, y1, x2, y2, ...], from manual/Smart Polygon labels) and
    RLE masks (dicts with 'counts', from auto-labeling), whose outline is traced from the mask."""
    if isinstance(seg, dict):
        mask = rle_to_mask(seg)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return [c.reshape(-1, 2).astype(np.float32) for c in contours
                if len(c) >= 3 and cv2.contourArea(c) >= 4]       # drop tiny specks
    return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg if len(p) >= 6]

def find_alignment(still, frame):
    """The stills are phone screenshots: a scaled, slightly cropped copy of the video frame.
    Try a range of scales and find where the scaled still best matches inside the frame."""
    fh, fw = frame.shape[:2]
    sh, sw = still.shape[:2]
    g_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    g_still = cv2.cvtColor(still, cv2.COLOR_BGR2GRAY)
    base = fw / sw                                             # screenshot width ~ full video width
    best = (-1.0, base, (0, 0))

    def search(scales):
        nonlocal best
        for scale in scales:
            w, h = int(round(sw * scale)), int(round(sh * scale))
            if w > fw or h > fh:                               # scaled still must fit inside the frame
                continue
            templ = cv2.resize(g_still, (w, h), interpolation=cv2.INTER_AREA)
            result = cv2.matchTemplate(g_frame, templ, cv2.TM_CCOEFF_NORMED)
            _, score, _, loc = cv2.minMaxLoc(result)
            if score > best[0]:
                best = (score, scale, loc)

    search(base * np.linspace(0.85, 1.02, 35))                 # coarse pass
    search(best[1] * np.linspace(0.995, 1.005, 21))            # fine pass around the best scale
    score, scale, (dx, dy) = best
    return score, scale, dx, dy

def read_empty_wall_frames(video, seconds=3.0, step=5):
    """Every 5th frame from the empty-wall part of the video.
    Cameras shake, adjust exposure and refocus at the start, so later empty-wall frames are often better."""
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    out = []
    for i in range(max(1, int(seconds * fps))):
        ok, f = cap.read()
        if not ok:
            break
        if i % step == 0:
            out.append((i, f))
    cap.release()
    return out

def best_frame(still, scale, candidates):
    """Pick the candidate frame that best matches the screenshot at the given scale (one quick match each)."""
    templ = cv2.cvtColor(cv2.resize(still, (int(round(still.shape[1] * scale)), int(round(still.shape[0] * scale))),
                                    interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    def quick(f):
        result = cv2.matchTemplate(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), templ, cv2.TM_CCOEFF_NORMED)
        return cv2.minMaxLoc(result)[1]
    return max(candidates, key=lambda c: quick(c[1]))

def to_video(parts, pos):
    """Convert outline points from screenshot pixels to video pixels."""
    scale, dx, dy = align[pos]
    return [(p * scale + np.array([dx, dy])).astype(np.float32) for p in parts]   # OpenCV needs float32

align, frames, checks = {}, {}, []
for key, im in images.items():
    pos = im['pos']
    print('aligning', pos, flush=True)
    videos = [find_video(v) for v in meta.loc[meta[POS_COL] == pos, VIDEO_COL]]
    video = next((v for v in videos if v), None)
    if video is None:
        checks.append({'position': pos, 'ok': 'NO VIDEO FOUND'})
        continue
    start = climb_start.get(os.path.splitext(os.path.basename(video))[0], 3.5)
    window = max(1.0, float(start) - 0.5) if pd.notna(start) else 3.0     # stop before the climber steps in
    candidates = read_empty_wall_frames(video, seconds=window)
    if not candidates:
        checks.append({'position': pos, 'ok': 'COULD NOT READ VIDEO', 'video_file': os.path.basename(video)})
        continue
    still = cv2.imread(im['path'])
    score, scale, dx, dy = find_alignment(still, candidates[0][1])       # first guess of the scale
    frame_idx, frame = best_frame(still, scale, candidates)             # least-changed empty-wall frame
    score, scale, dx, dy = find_alignment(still, frame)                 # refine on that frame
    align[pos], frames[pos] = (scale, dx, dy), frame
    checks.append({'position': pos, 'still': f"{im['width']}x{im['height']}",
                   'video': f'{frame.shape[1]}x{frame.shape[0]}', 'scale': round(scale, 4),
                   'offset': (dx, dy), 'match_score': round(score, 3), 'ok': score >= 0.8,
                   'frame_used': frame_idx,
                   'video_file': os.path.basename(video)})
    overlay = frame.copy()                                     # converted outlines on the video frame
    for a in anns_by_img[key]:
        for p in to_video(polygon_parts(a['segmentation']), pos):
            cv2.polylines(overlay, [p.astype(np.int32)], True, (0, 255, 255), 2)
    cv2.imwrite(f'{MAP_DIR}/checks/{pos}_overlay.jpg', overlay)
    cv2.imwrite(f'{MAP_DIR}/{pos}_frame.png', frame)            # clean reference frame (video coordinates) for later steps

display(pd.DataFrame(checks).sort_values('position'))

## Cell 5: Build the hold table
Reads both label formats: polygons and RLE masks (from auto-labeling). All coordinates are converted to **video pixels**. IDs go bottom to top, then left to right. Saves `holds.csv` and `hold_polygons.json`.
Positions whose alignment failed in Cell 4 are skipped.

In [ ]:
def parse_class(name):
    """'red-hold' or 'hold-red' -> ('hold', 'red'); 'down' -> ('down', ''); 'volume' -> ('volume', '')."""
    n = name.lower()
    if n.endswith('-hold'):
        return 'hold', n[:-len('-hold')]
    if n.startswith('hold-'):
        return 'hold', n[len('hold-'):]
    return n, ''

hold_rows, polygons, items_by_pos = [], {}, {}
for key, im in images.items():
    pos = im['pos']
    if pos not in align:
        print(f'Skipping {pos}: no aligned video (see Cell 4)')
        continue
    items = []
    for a in anns_by_img[key]:
        parts = to_video(polygon_parts(a['segmentation']), pos)     # all coordinates in video pixels
        if not parts:
            continue
        pts = np.concatenate(parts)
        x0, y0 = pts.min(axis=0)
        x1, y1 = pts.max(axis=0)
        areas = [cv2.contourArea(p) for p in parts]
        M = cv2.moments(parts[int(np.argmax(areas))])          # center of the largest part
        cx, cy = (M['m10'] / M['m00'], M['m01'] / M['m00']) if M['m00'] else ((x0 + x1) / 2, (y0 + y1) / 2)
        cls, color = parse_class(cats[a['category_id']])
        items.append({'cls': cls, 'color': color, 'cx': cx, 'cy': cy, 'parts': parts,
                      'bbox': (x0, y0, x1 - x0, y1 - y0), 'area': sum(areas)})

    items.sort(key=lambda d: (-d['cy'], d['cx']))              # bottom to top, then left to right
    for hold_id, d in enumerate(items, start=1):
        d['hold_id'] = hold_id
        bx, by, bw, bh = d['bbox']
        hold_rows.append({'position_id': pos, 'hold_id': hold_id, 'cls': d['cls'], 'color': d['color'],
                          'center_x': round(float(d['cx']), 1), 'center_y': round(float(d['cy']), 1),
                          'bbox_x': round(float(bx), 1), 'bbox_y': round(float(by), 1),
                          'width': round(float(bw), 1), 'height': round(float(bh), 1),
                          'area': round(float(d['area']), 1)})
        polygons.setdefault(pos, {})[str(hold_id)] = [p.round(1).tolist() for p in d['parts']]
    items_by_pos[pos] = items

holds_df = pd.DataFrame(hold_rows).sort_values(['position_id', 'hold_id'])
n_labels = sum(len(anns_by_img[k]) for k, im in images.items() if im['pos'] in align)
print(f'{n_labels} labels at aligned positions -> {len(holds_df)} holds in the table')   # should be (nearly) equal
holds_df.to_csv(f'{MAP_DIR}/holds.csv', index=False)
json.dump(polygons, open(f'{MAP_DIR}/hold_polygons.json', 'w'))
print(holds_df.groupby('position_id').size().rename('holds').to_string())

## Cell 6: Draw the numbered images
Drawn on each position's video frame. Yellow outlines = holds, magenta = volumes, blue = down-climb holds

In [ ]:
def draw_numbered(img, items):
    out = img.copy()
    s = max(img.shape[:2]) / 1500                              # scale lines and text to image size
    for d in items:
        color = {'hold': (0, 255, 255), 'volume': (255, 0, 255)}.get(d['cls'], (255, 160, 0))
        for p in d['parts']:
            cv2.polylines(out, [p.astype(np.int32)], True, color, max(1, int(2 * s)))
    for d in items:
        txt, font, fs = str(d['hold_id']), cv2.FONT_HERSHEY_SIMPLEX, 0.7 * s
        (tw, th), _ = cv2.getTextSize(txt, font, fs, max(1, int(2 * s)))
        org = (int(d['cx'] - tw / 2), int(d['cy'] + th / 2))   # center the number on the hold
        cv2.putText(out, txt, org, font, fs, (0, 0, 0), max(3, int(5 * s)), cv2.LINE_AA)       # dark outline
        cv2.putText(out, txt, org, font, fs, (255, 255, 255), max(1, int(2 * s)), cv2.LINE_AA)  # white text
    return out

for pos, items in items_by_pos.items():                        # drawn on the video frame, in video coordinates
    cv2.imwrite(f'{MAP_DIR}/{pos}_numbered.png', draw_numbered(frames[pos], items))
print('Saved numbered images to', MAP_DIR)

## Cell 7: Pre-fill `routes.csv`
Adds, for each route at a labeled position, every hold of the route's color and every volume.

In [ ]:
ROUTES_CSV = f'{MAP_DIR}/routes.csv'
COLORS = sorted(c for c in holds_df.loc[holds_df.cls == 'hold', 'color'].unique() if c)   # never match uncolored holds

existing = pd.read_csv(ROUTES_CSV) if os.path.exists(ROUTES_CSV) else pd.DataFrame(
    columns=['route_id', 'position_id', 'hold_id', 'cls', 'color', 'on_route', 'role'])
done = set(zip(existing.route_id.astype(str), existing.position_id.astype(str)))

new_rows = []
for route, pos, color_text in meta[['route_id', POS_COL, 'route_color']].drop_duplicates().itertuples(index=False):
    if pos not in items_by_pos or (route, pos) in done:        # not labeled yet, or already filled in
        continue
    words = re.split(r'[^a-z]+', color_text)
    color = next((c for c in COLORS if c in words), None)      # match against your label colors
    if color is None:
        print(f'Could not match color "{color_text}" for route {route}; fill its holds in by hand.')
    for h in holds_df[holds_df.position_id == pos].itertuples():
        if h.cls == 'hold' and h.color == color:
            new_rows.append([route, pos, h.hold_id, h.cls, h.color, 1, 'middle'])
        elif h.cls == 'volume':
            new_rows.append([route, pos, h.hold_id, h.cls, '', '', ''])   # decide by hand

routes = pd.concat([existing, pd.DataFrame(new_rows, columns=existing.columns)], ignore_index=True)
routes.to_csv(ROUTES_CSV, index=False)
print(f'Added {len(new_rows)} rows for {len({(r[0], r[1]) for r in new_rows})} route/position pairs')